# 🐱🐹 KittenTTS-Go · Quick Test on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itamaker/kitten-tts-go/blob/main/kitten-tts-go-colab.ipynb)

Try [`itamaker/kitten-tts-go`](https://github.com/itamaker/kitten-tts-go) on Colab (Ubuntu / x86_64, **no GPU needed**) — an ultra-lightweight, ONNX-based, pure-Go TTS engine.

**① Setup → ② Generate → ③ Try it out** — one cell each, run top to bottom.

## ① Setup

Install dependencies (espeak-ng, libopus, Go, ONNX Runtime) and clone the repo. ONNX Runtime is pinned to 1.25.0 to match the Go bindings' C API.

In [ ]:
%env CGO_ENABLED=1
%env ONNXRUNTIME_LIB_PATH=/usr/local/lib/libonnxruntime.so.1.25.0

!sudo apt-get update -qq && sudo apt-get install -y -qq espeak-ng libopus-dev libopusfile-dev pkg-config
!wget -q https://go.dev/dl/go1.23.4.linux-amd64.tar.gz && sudo rm -rf /usr/local/go && sudo tar -C /usr/local -xzf go1.23.4.linux-amd64.tar.gz && sudo ln -sf /usr/local/go/bin/go /usr/local/bin/go
!wget -q https://github.com/microsoft/onnxruntime/releases/download/v1.25.0/onnxruntime-linux-x64-1.25.0.tgz && tar -xzf onnxruntime-linux-x64-1.25.0.tgz && sudo cp -P onnxruntime-linux-x64-1.25.0/lib/libonnxruntime.so* /usr/local/lib/
!git clone --depth 1 https://github.com/itamaker/kitten-tts-go.git
%cd kitten-tts-go

## ② Generate

Download the model (nano-int8, ~25 MB), build the binaries, and synthesize `hello.wav`. Go `flag` convention: flags come before the positional args `<model_dir> <text> [voice]`.

In [ ]:
!scripts/fetch_model.sh nano-int8
!go build -o bin/ ./...
!./bin/kitten-tts -voice Luna -output /content/hello.wav \
    ./models/kitten-tts-nano-int8 'Hello, world! This is KittenTTS running in Google Colab.'

## ③ Try it out

See the 8 built-in voices and play the clip you just generated. Want a different voice / speed / format? Edit the command in ② (e.g. `-voice Bruno -speed 1.2 -format mp3`) and re-run.

In [ ]:
!./bin/kitten-tts -list-voices ./models/kitten-tts-nano-int8
from IPython.display import Audio
Audio('/content/hello.wav')

**(Optional) OpenAI-compatible API server** — two steps: **Step 1** starts the server on port 8888 (`nohup … &`), **Step 2** sends a `curl` request. Edit any field below and re-run Step 2 as much as you like.

Request body fields (every supported param):

| Field | Default | Notes |
|---|---|---|
| `model` | — | accepted for compatibility, ignored |
| `input` | *(required)* | text to synthesize |
| `voice` | *(required)* | KittenTTS name: Bella / Jasper / Luna / Bruno / Rosie / Hugo / Kiki / Leo |
| `response_format` | `mp3` | `mp3` / `wav` / `flac` / `opus` / `pcm` |
| `speed` | `1.0` | 0.25–4.0 |
| `stream` | `false` | SSE streaming; requires `response_format: "pcm"` |

OpenAI-compatible: point any OpenAI TTS client at this endpoint by just changing the base URL — OpenAI voice names (alloy, echo, fable, onyx, nova, shimmer) are also accepted and mapped to KittenTTS voices.

In [ ]:
# Step 1 — start the server on port 8888
!nohup ./bin/kitten-tts-server -host 127.0.0.1 -port 8888 ./models/kitten-tts-nano-int8 > /content/server.log 2>&1 &
!sleep 5; cat /content/server.log

In [ ]:
# Step 2 — request speech (edit any field and re-run as you like)
!curl -sS http://127.0.0.1:8888/v1/audio/speech \
    -H 'Content-Type: application/json' \
    -d '{"model":"kitten-tts","input":"Hello from the KittenTTS API server.","voice":"Luna","response_format":"mp3","speed":1.0,"stream":false}' \
    --output /content/api.mp3
from IPython.display import Audio
Audio('/content/api.mp3')